In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import gc

# Clear GPU cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Force garbage collection
gc.collect()

print("Cache cleared and garbage collected!")

Cache cleared and garbage collected!


In [3]:
proj_path = "/content/drive/MyDrive/llm_from_scratch/src"

In [4]:
import os
os.chdir(proj_path)
print(os.getcwd())

/content/drive/MyDrive/llm_from_scratch/src


In [5]:
from loading_weights import gpt as model

File already exists and is up-to-date: gpt2/774M/checkpoint
File already exists and is up-to-date: gpt2/774M/encoder.json
File already exists and is up-to-date: gpt2/774M/hparams.json
File already exists and is up-to-date: gpt2/774M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/774M/model.ckpt.index
File already exists and is up-to-date: gpt2/774M/model.ckpt.meta
File already exists and is up-to-date: gpt2/774M/vocab.bpe


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
import tiktoken
tokenizer = tiktoken.get_encoding('gpt2')

In [8]:
from dataloaders_for_classification_pretraining import train_dataloader as train_loader, validation_dataloader as val_loader, test_dataloader as test_loader


In [9]:
for param in model.parameters():
    param.requires_grad = False

In [10]:
torch.manual_seed(123)
num_classes = 2
model.out_head = torch.nn.Linear(
in_features=1280,
out_features=num_classes
)

In [11]:
for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True
for param in model.final_norm.parameters():
    param.requires_grad = True

In [12]:
model = model.to(device)
print(device)

cuda


In [40]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    pooled_logits = logits.mean(dim=1)

    # For binary classification
    loss = torch.nn.functional.binary_cross_entropy_with_logits(
        pooled_logits[:, 1],  # Use the spam class logits
        target_batch.float()
    )
    return loss

In [25]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [26]:
def train_classifier_simple(model, train_loader, val_loader, optimizer, device,
                            num_epochs, eval_freq, eval_iter, tokenizer):
    # initialize lists to track losses and examples seen
    train_losses, val_losses = [], []
    examples_seen, global_step = 0, -1
    # main training loop
    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            examples_seen += input_batch.shape[0]
            global_step += 1
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")
    return train_losses, val_losses, examples_seen

In [27]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device,
                                      num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

In [29]:
import time
start_time = time.time()
torch.manual_seed(123)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
num_epochs = 3
train_losses, val_losses, examples_seen = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=50, eval_iter=5,
    tokenizer=tokenizer
)
end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

Ep 1 (Step 000000): Train loss 0.690, Val loss 0.629
Ep 1 (Step 000050): Train loss 0.466, Val loss 0.416
Ep 1 (Step 000100): Train loss 0.115, Val loss 0.184
Ep 2 (Step 000150): Train loss 0.063, Val loss 0.047
Ep 2 (Step 000200): Train loss 0.018, Val loss 0.063
Ep 3 (Step 000250): Train loss 0.020, Val loss 0.098
Ep 3 (Step 000300): Train loss 0.010, Val loss 0.025
Training completed in 4.69 minutes.


In [18]:
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 1280)
  (pos_emb): Embedding(1024, 1280)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (dropout): Dropout(p=0.1, inplace=False)
        (W_query): Linear(in_features=1280, out_features=1280, bias=True)
        (W_key): Linear(in_features=1280, out_features=1280, bias=True)
        (W_value): Linear(in_features=1280, out_features=1280, bias=True)
        (out_proj): Linear(in_features=1280, out_features=1280, bias=True)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1280, out_features=5120, bias=True)
          (1): GELU()
          (2): Linear(in_features=5120, out_features=1280, bias=True)
        )
      )
      (norm1): LayerNormalization()
      (norm2): LayerNormalization()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (

In [30]:
from function_to_classify_text import classify_review

In [62]:
text_1 = (
"hello when i will pick you from home and you won a prize money of 10k?"
)

In [63]:
result= classify_review(text_1, model, tokenizer, device, max_length=50)

In [64]:
print(result)

spam


In [65]:
# os.makedirs("/content/drive/MyDrive/llm_from_scratch/classification_model", exist_ok=True)

In [74]:
# torch.save(model.state_dict(), "/content/drive/MyDrive/llm_from_scratch/classification_model/spam_classifier_state.pth")
# print("Model saved!")

Model saved!
